# YT Agent — Colab GPU backend

This notebook spins up the FastAPI backend on a free Colab GPU runtime, exposes it via a Cloudflare quick tunnel, and registers itself in the Hostinger registry so your Vercel frontend can find it.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (free).
2. Open **🔑 Secrets** (left sidebar key icon) and add all the secrets listed in `colab/secrets.example` from the repo.
3. Run cells top to bottom. The last cell is a blocking server loop — leave it running.

When the tunnel URL appears, the registry update is automatic. Your Vercel dashboard will pick this backend up within ~30s.

In [ ]:
# 1) System deps
import subprocess, sys, os
subprocess.check_call(["apt-get", "-qq", "update"])
subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])
print("ffmpeg:", subprocess.check_output(["ffmpeg", "-version"]).decode().splitlines()[0])

In [ ]:
# 2) Clone the repo (or pull latest if already present)
REPO_URL = "https://github.com/YOUR_GITHUB_USER/yt_agent.git"  # ← edit before first run
BRANCH   = "main"
import os, subprocess
if not os.path.exists('/content/yt_agent'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/yt_agent'])
else:
    subprocess.check_call(['git', '-C', '/content/yt_agent', 'pull', '--ff-only'])
os.chdir('/content/yt_agent')
print('repo at:', os.getcwd())

In [ ]:
# 3) Python deps
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
# 4) Pull secrets from Colab Secrets into the process environment.
#    Add these in the left sidebar: 🔑 Secrets icon.
from google.colab import userdata
import os

REQUIRED = [
    'NVIDIA_NIM_API_KEY',
    'SHUTTERSTOCK_API_TOKEN',
    'PEXELS_API_KEY',
    'PIXABAY_API_KEY',
    # FTP / Hostinger storage
    'FTP_HOST', 'FTP_USER', 'FTP_PASS',
    'PUBLIC_BASE_URL',
]
OPTIONAL = [
    'COVERR_API_KEY', 'SHUTTERSTOCK_CLIENT_ID', 'SHUTTERSTOCK_CLIENT_SECRET',
    'GROQ_API_KEY', 'FTP_BASE_DIR', 'FTP_PORT', 'FTP_USE_TLS',
]

missing = []
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = str(v)
    except Exception:
        if k in REQUIRED:
            missing.append(k)
        continue
    if k in REQUIRED and not os.environ.get(k):
        missing.append(k)

if missing:
    print('MISSING REQUIRED SECRETS:', ', '.join(missing))
    print('Add them in the 🔑 Secrets panel, then re-run this cell.')
else:
    print('All required secrets loaded.')

In [ ]:
# 5) Cloudflare quick tunnel — free, no auth. Returns a stable URL
#    until the Colab session ends.
import os, subprocess, time, re, threading

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.check_call([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared',
    ])
    subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])

# Start the tunnel to localhost:8000 (where uvicorn will run in the next cell)
tunnel_log = '/tmp/cloudflared.log'
subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
     '--logfile', tunnel_log, '--loglevel', 'info'],
    stdout=open(tunnel_log + '.stdout', 'w'),
    stderr=subprocess.STDOUT,
)

# Wait for the URL to show up in the log (~5-10s)
url = None
for _ in range(40):
    time.sleep(0.5)
    if not os.path.exists(tunnel_log):
        continue
    with open(tunnel_log) as f:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
        if m:
            url = m.group(0)
            break
if not url:
    raise RuntimeError('cloudflared did not produce a URL — check /tmp/cloudflared.log')

os.environ['PUBLIC_BACKEND_URL'] = url
print('Public backend URL:', url)

In [ ]:
# 6) Boot the backend. This cell BLOCKS — keep it running for the session.
#    The heartbeat (registry.start) auto-registers PUBLIC_BACKEND_URL into
#    your Hostinger registry.json every 30s.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])